In [30]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv('/content/drive/MyDrive/dataset/car_prediction_data.csv')
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [5]:
df.tail()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
296,city,2016,9.50,11.6,33988,Diesel,Dealer,Manual,0
297,brio,2015,4.00,5.9,60000,Petrol,Dealer,Manual,0
298,city,2009,3.35,11.0,87934,Petrol,Dealer,Manual,0
299,city,2017,11.50,12.5,9000,Diesel,Dealer,Manual,0
300,brio,2016,5.30,5.9,5464,Petrol,Dealer,Manual,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Seller_Type    301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB


In [14]:
print("--- Step 1: Initial Dataset Exploration ---")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Missing values per column:")
print(df.isnull().sum())
print("\nFirst 5 rows of the dataset:")
print(df.head())
print("-" * 50)

--- Step 1: Initial Dataset Exploration ---
Dataset Dimensions: 299 rows, 9 columns

Missing values per column:
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
dtype: int64

First 5 rows of the dataset:
  Car_Name  Year  Selling_Price  Present_Price  Kms_Driven Fuel_Type  \
0     ritz  2014           3.35           5.59       27000    Petrol   
1      sx4  2013           4.75           9.54       43000    Diesel   
2     ciaz  2017           7.25           9.85        6900    Petrol   
3  wagon r  2011           2.85           4.15        5200    Petrol   
4    swift  2014           4.60           6.87       42450    Diesel   

  Seller_Type Transmission  Owner  
0      Dealer       Manual      0  
1      Dealer       Manual      0  
2      Dealer       Manual      0  
3      Dealer       Manual      0  
4      Dealer       Manual      0  
-------------------------

In [15]:
#removing duplicate
duplicate_count = df.duplicated().sum()
print(f"--- Step 2: Duplicate Analysis ---")
print(f"Found {duplicate_count} duplicate records.")
if duplicate_count > 0:
    df = df.drop_duplicates()
    df.reset_index(drop=True, inplace=True)
    print("Duplicate records successfully removed.")
print("-" * 50)

--- Step 2: Duplicate Analysis ---
Found 0 duplicate records.
--------------------------------------------------


In [16]:
#feature engineering
print("--- Step 3: Feature Engineering ---")
# Calculate the Age of the car using the current year (2026)
current_year = 2026
df['Vehicle_Age'] = current_year - df['Year']

# Drop columns that are not useful for prediction
# - 'Car_Name': High cardinality text column (too many unique names)
# - 'Year': Replaced by the engineered 'Vehicle_Age' column
columns_to_drop = ['Car_Name', 'Year']
df = df.drop(columns=columns_to_drop)
print(f"Created 'Vehicle_Age' column and dropped original columns: {columns_to_drop}")
print("-" * 50)

--- Step 3: Feature Engineering ---
Created 'Vehicle_Age' column and dropped original columns: ['Car_Name', 'Year']
--------------------------------------------------


In [17]:
df.head()

,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Vehicle_Age
0,3.35,5.59,27000,Petrol,Dealer,Manual,0,12
1,4.75,9.54,43000,Diesel,Dealer,Manual,0,13
2,7.25,9.85,6900,Petrol,Dealer,Manual,0,9
3,2.85,4.15,5200,Petrol,Dealer,Manual,0,15
4,4.60,6.87,42450,Diesel,Dealer,Manual,0,12


In [20]:
#outlier treatment
print("--- Step 4: Outlier Treatment ---")
# Continuous numerical columns likely to contain extreme outliers
numerical_outlier_cols = ['Kms_Driven', 'Present_Price']

for col in numerical_outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap (clip) outliers to keep the data size intact while preventing skewness
    df[col] = np.clip(df[col], lower_bound, upper_bound)
    print(f"Applied IQR capping to '{col}' (Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f})")
print("-" * 50)

--- Step 4: Outlier Treatment ---
Applied IQR capping to 'Kms_Driven' (Lower: -35825.25, Upper: 99708.75)
Applied IQR capping to 'Present_Price' (Lower: -11.76, Upper: 22.80)
--------------------------------------------------


In [21]:
#categorical encoding
print("--- Step 5: Encoding Categorical Variables ---")
# Identifies nominal categorical variables
categorical_cols = ['Fuel_Type', 'Seller_Type', 'Transmission']

# Apply One-Hot Encoding and drop the first category to prevent multicollinearity (dummy variable trap)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Applied One-Hot Encoding. Current columns in dataset:")
print(df.columns.tolist())
print("-" * 50)

--- Step 5: Encoding Categorical Variables ---
Applied One-Hot Encoding. Current columns in dataset:
['Selling_Price', 'Present_Price', 'Kms_Driven', 'Owner', 'Vehicle_Age', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Seller_Type_Individual', 'Transmission_Manual']
--------------------------------------------------


In [22]:
df.head()

,Selling_Price,Present_Price,Kms_Driven,Owner,Vehicle_Age,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
0,3.35,5.59,27000.0,0,12,False,True,False,True
1,4.75,9.54,43000.0,0,13,True,False,False,True
2,7.25,9.85,6900.0,0,9,False,True,False,True
3,2.85,4.15,5200.0,0,15,False,True,False,True
4,4.60,6.87,42450.0,0,12,True,False,False,True


In [25]:
#train test
print("--- Step 6: Train-Test Split ---")
# Separate independent features (X) and target variable (y)
target_col = 'Selling_Price'
X = df.drop(columns=[target_col])
y = df[target_col]

# Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set features shape: {X_train.shape}")
print(f"Testing set features shape: {X_test.shape}")
print("-" * 50)

--- Step 6: Train-Test Split ---
Training set features shape: (239, 8)
Testing set features shape: (60, 8)
--------------------------------------------------


In [28]:
#missing value imputation
print("--- Step 7: Missing Value Imputation ---")
# While the original dataset may be clean, we run an imputer to make the pipeline robust to future real-world data
imputer = SimpleImputer(strategy='median')

# Fit on training data and transform both train and test sets to prevent leakage
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)
print("Missing values handled using Median Imputation.")
print("-" * 50)

--- Step 7: Missing Value Imputation ---
Missing values handled using Median Imputation.
--------------------------------------------------


In [31]:
#feature scaling
print("--- Step 8: Feature Scaling ---")
# Scale numerical columns so that distance-based regression models function correctly
scaler = StandardScaler()

# Fit on training data and scale both training and testing datasets
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train_imputed.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test_imputed.columns)
print("Features scaled using StandardScaler (Mean = 0, Standard Deviation = 1).")
print("-" * 50)

--- Step 8: Feature Scaling ---
Features scaled using StandardScaler (Mean = 0, Standard Deviation = 1).
--------------------------------------------------


In [32]:
print("--- Step 9: Class Imbalance Evaluation ---")
print("Task Type: Regression")
print("Decision: SMOTE-Tomek was NOT applied.")
print("Justification: Since the target variable 'Selling_Price' is continuous, resampling methods ")
print("like SMOTE-Tomek (which are strictly designed for categorical target classes) are not applicable.")
print("=" * 50)
print("PREPROCESSING PIPELINE SUCCESSFULLY COMPLETED!")

--- Step 9: Class Imbalance Evaluation ---
Task Type: Regression
Decision: SMOTE-Tomek was NOT applied.
Justification: Since the target variable 'Selling_Price' is continuous, resampling methods 
like SMOTE-Tomek (which are strictly designed for categorical target classes) are not applicable.
PREPROCESSING PIPELINE SUCCESSFULLY COMPLETED!
